Proceso de flujo contínuo

Frecuencia de muestreo de 1 Hz

Máquinas 1, 2 y 3 funcionan en paralelo. Sus salidas se combinan en un flujo

La salida del combinador se mide en 15 lugares distintos. Son los primeros parámetros a medir.

La salida entra en la máquina 4, y la 5 está a continuación.



In [ ]:
import pandas as pd
import numpy as np


In [ ]:
filepath = '../Datasets/continuous_factory_process.csv'

TARGET = "FirstStage.CombinerOperation.Temperature2.U.Actual"

df = pd.read_csv(filepath)
df.head(-5)

df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df = df.set_index('time_stamp').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.asfreq('1s')

duración del proceso = 3 h 55 min aprox

variables por columna:
0	0	Time stamp

1	2	Factory ambient conditions

3	6	First stage, Machine 1, raw material properties (material going in to Machine 1)

7	14	First stage, Machine 1 process variables

15	18	First stage, Machine 2, raw material properties (material going in to Machine 2)

19	26	First stage, Machine 2 process variables

27	30	First stage, Machine 3, raw material properties (material going in to Machine 3)

31	38	First stage, Machine 3 process variables

39	41	Combiner stage process parameters. Here we combines the outputs from Machines 1, 2, and 3.

42	71	PRIMARY OUTPUT TO CONTROL: Measurements of 15 features (in mm), along with setpoint or target for each

72	78	Second stage, Machine 4 process variables

79	85	Second stage, Machine 5 process variables

86	115	SECONDARY OUTPUT TO CONTROL: Measurements of 15 features (in mm), along with setpoint or target for each

---------------------------------------

delay entre la entrada de material a las máquinas y la salida; entrada del combinador a la salida del combinador; salida del combinador hasta la medida

se supone que variables que tienen muy poca variación (pasa con la temperatura) pueden ser rechazadas

In [ ]:
# Remove columns which contains "Stage2"
bad_words = ["Stage2", "Machine4", "Machine5", "Setpoint"]

df = df.drop(columns=[col for col in df.columns if any(word in col for word in bad_words)])
df = df.drop(columns=[col for col in df.columns if "Stage1.Output.Measurement" in col and col != TARGET])
print(df.columns.tolist())

Check missing values


In [ ]:
import pandas as pd

empty_list = []

for column in df.columns:
    is_null = df[column].isna()
    
    if not is_null.any():
        continue

    blocks = (~is_null).cumsum()
    null_dates = df[is_null].reset_index()
    time_col = null_dates.columns[0] 
    
    rangos = null_dates.groupby(blocks[is_null].values)[time_col].agg(
        Inicio='min',
        Fin='max'
    )
    
    rangos['Variable'] = column
    
    # 1. Calcular la duración y convertirla a segundos (número entero)
    rangos['Duracion_segundos'] = (rangos['Fin'] - rangos['Inicio']).dt.total_seconds().astype(int)
    
    # 2. Formatear las fechas para que siempre tengan la misma longitud y se alineen bien
    rangos['Inicio'] = rangos['Inicio'].dt.strftime('%Y-%m-%d %H:%M:%S')
    rangos['Fin'] = rangos['Fin'].dt.strftime('%Y-%m-%d %H:%M:%S')
    
    empty_list.append(rangos)

if empty_list:
    df_nulos = pd.concat(empty_list).reset_index(drop=True)
    df_nulos = df_nulos[['Variable', 'Inicio', 'Fin', 'Duracion_segundos']]
    
    # Imprimir forzando la alineación para que se lea perfecto en la consola
    print(df_nulos.to_string(justify='center'))
else:
    df_nulos = pd.DataFrame()
    print("No hay valores nulos en el DataFrame.")

Fill empty values using the mean

In [ ]:
for column in df.columns:
    if df[column].isna().any():
        df[column] = df[column].interpolate(method='linear', limit_direction='both')

# Check if still there are any null values after interpolation
for column in df.columns:
    if df[column].isna().any():
        print(f"Warning: Column '{column}' still has null values after interpolation.")

Statistical analysis

In [ ]:
# 1. Seleccionar solo las columnas numéricas (evita errores con fechas o texto)
df_num = df.select_dtypes(include=['number'])

# 2. Generar las estadísticas básicas y transponer (.T) para que las variables sean filas
stats = df_num.describe().T

# 3. Calcular el Coeficiente de Variación (CV = std / mean)
# Lo multiplicamos por 100 para expresarlo como porcentaje
stats['CV (%)'] = (stats['std'] / stats['mean']) * 100

# 4. (Opcional) Limpiar el formato para que sea más fácil de leer
# Redondeamos a 2 decimales para no tener números gigantes en pantalla
stats = stats.round(2)

# Mostrar la tabla final
print(stats.to_string())

In [ ]:
# Export dataset to csv. Insert the time_stamp column at the left and then export to csv. Then set again the time_stamp column as index
df.insert(0, "time_stamp", df.index)
df.to_csv("continuous_process_filtered.csv", index=False)
df.set_index("time_stamp", inplace=True)

# Observe data distribution with histograms (tsbox)
Máquina 1
- La distribución del FeederParameter de la máquina 1 tiene una única distribución parece que bastante expandida por ruido. Habría que hacer un filtrado para achatar la gausiana
Las señales 
- No vamos a usar la  

# Drop signals whose CV is under 1

In [ ]:
# 1. Seleccionar solo columnas numéricas
df_num = df.select_dtypes(include=[np.number])

# 2. Calcular el Coeficiente de Variación (CV = std / mean)
# Se usa np.abs por si la media fuera negativa
cv_series = (df_num.std() / df_num.mean()).abs()

# 3. Filtrar columnas donde CV < 1
columnas_cv_menor_1 = cv_series[cv_series < 1]

# 4. Mostrar el DataFrame resultante solo con esas columnas
df_filtrado = df[columnas_cv_menor_1.index]

print("--- Columnas con CV < 1 y su valor ---")
print(columnas_cv_menor_1)

print("\n--- Visualización de las columnas filtradas ---")
print(df_filtrado.head())

Análisis de señales gráfico

In [ ]:
import matplotlib.pyplot as plt

# 1. ESCRIBE AQUÍ LAS SEÑALES QUE QUIERES VER (Deben existir en tu DataFrame 'df')
# Ejemplo: señales_a_pintar = ["Stage2_Temp", "Machine4_Pressure"]
señales_a_pintar = df.columns.tolist()  # Esto pintará todas las señales, puedes cambiarlo a una lista específica 

# Comprobar que hemos puesto alguna señal
if not señales_a_pintar:
    print("Por favor, añade al menos una señal a la lista 'señales_a_pintar'.")
else:
    # 2. Crear las subfiguras (una debajo de otra)
    # Ajustamos la altura (2.5) multiplicada por el número de señales
    fig, axes = plt.subplots(
        nrows=len(señales_a_pintar), 
        ncols=1, 
        figsize=(12, 2.5 * len(señales_a_pintar)), 
        sharex=True
    )
    
    # Ajuste por si solo pones 1 señal en la lista (para que no falle el bucle)
    if len(señales_a_pintar) == 1:
        axes = [axes]
        
    # 3. Pintar cada señal en su gráfico correspondiente
    for ax, columna in zip(axes, señales_a_pintar):
        # Verifica que la columna exista para evitar errores tontos
        if columna in df.columns:
            ax.plot(df.index, df[columna], color='#1f77b4', linewidth=1.5)
            ax.set_title(columna, fontsize=12, fontweight='bold', loc='left')
            ax.grid(True, linestyle='--', alpha=0.7)
            # Ocultar el marco superior y derecho para que quede más limpio
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        else:
            ax.text(0.5, 0.5, f"La columna '{columna}' no existe", 
                    ha='center', va='center', color='red')
            
    # Ajustar espacios para que no se superpongan los títulos con los ejes
    plt.tight_layout()
    plt.show()

# TODO after visualization
- Analizar estadísticamente desde el inicio de la señal hasta la hora 11:11. Hacer otro análisis aparte desde las 11:11 hasta el final
- Machine2.MotorRPM.C.Actual no aporta
- Machine2.Zone1Temperature parece que tampoco
- Machine2.Zone2Temperature parece que tampoco
- Machine2.ExitZoneTemperature mantiene la media, pero la varianza cambia con el paso del tiempo

- Media movil 
-- Machine2.MaterialPressure
-- Machine2.MotorAmperage

Training split

## Diferenciación del target (estacionariedad)

`TARGET` en nivel (°C absolutos) tiene un desplazamiento fuerte de régimen a lo largo
de la campaña: train cae mayormente en 53-115°C (media 76.5, std 16.1, incluye la
rampa de transición), mientras que val y test caen casi enteros en una meseta alta y
estrecha (~102-108°C, std ~0.6). Train y val/test casi no se solapan en nivel — ningún
modelo puede generalizar a un régimen que apenas vio en entrenamiento.

Diferenciando (`Δy[t] = y[t] - y[t-1]`) se elimina el nivel y el modelo predice el
cambio, no el valor absoluto. Tras diferenciar, la media es ≈0 en los tres splits
(antes las medias de nivel eran completamente distintas), aunque el std de train sigue
siendo ~4.5x mayor que en val/test (train incluye la rampa con saltos de hasta ±61°C/s).
No es estacionariedad perfecta, pero es un salto grande respecto al nivel absoluto.

El baseline naive para una serie diferenciada es predecir Δ=0 (equivalente a la
persistencia y[t]=y[t-1] en la serie original).

In [ ]:
# Serie diferenciada del target. La primera fila queda NaN (no hay t-1 anterior) y se descarta.
df['TARGET_diff'] = df[TARGET].diff()
df = df.iloc[1:].copy()

print(f"Filas tras descartar la primera (sin diff válido): {len(df)}")
print(df[['TARGET_diff']].describe())

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler

target_col = 'TARGET_diff'
# Excluimos tanto el nivel (TARGET) como el propio diff de las features: el nivel
# porque es justo lo que estamos evitando (regimen no estacionario), y el diff
# porque es el target.
feature_cols = [c for c in df.columns if c not in ['time_stamp', TARGET, target_col]]

X_raw = df[feature_cols].values
y_raw = df[[target_col]].values

# 2. Split cronológico (70% Train, 15% Val, 15% Test)
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train_raw, y_train_raw = X_raw[:train_end], y_raw[:train_end]
X_val_raw, y_val_raw     = X_raw[train_end:val_end], y_raw[train_end:val_end]
X_test_raw, y_test_raw   = X_raw[val_end:], y_raw[val_end:]

# 3. Escalado independiente (Fit solo en Train)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = scaler_X.fit_transform(X_train_raw)
y_train = scaler_y.fit_transform(y_train_raw)

X_val = scaler_X.transform(X_val_raw)
y_val = scaler_y.transform(y_val_raw)

X_test = scaler_X.transform(X_test_raw)
y_test = scaler_y.transform(y_test_raw)

## Sequence windowing

LSTM necesita ventanas de secuencia (no filas sueltas). Cada muestra de entrada es una ventana de `SEQ_LEN` pasos de tiempo de todas las features, y el target es el valor de `TARGET` en el siguiente paso.

Usamos los arrays ya escalados (`X_train`, `X_val`, `X_test`) del split cronológico anterior — el escalado fue ajustado solo en train, así que no hay fuga de información (data leakage).

In [ ]:
SEQ_LEN = 10   # 10 s de historia -- reducido desde 60s: el target diferenciado
               # cambia rápido (casi ruido segundo a segundo), así que una ventana
               # larga diluye la dinámica reciente relevante con historia antigua
               # menos útil.
BATCH_SIZE = 64

class SequenceDataset(Dataset):
    """Ventanas deslizantes: X[t-SEQ_LEN:t] -> y[t]."""
    def __init__(self, X, y, seq_len):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        x_seq = self.X[idx: idx + self.seq_len]
        y_target = self.y[idx + self.seq_len]
        return x_seq, y_target

train_ds = SequenceDataset(X_train, y_train, SEQ_LEN)
val_ds   = SequenceDataset(X_val,   y_val,   SEQ_LEN)
test_ds  = SequenceDataset(X_test,  y_test,  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Loader adicional para evaluar sobre train SIN barajar: necesario para que las
# predicciones salgan en el mismo orden temporal que 'y_train' y se puedan comparar
# 1 a 1 con el baseline naive y los timestamps. 'train_loader' (shuffle=True) solo
# se usa para entrenar, nunca para evaluar/predecir.
train_eval_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train windows: {len(train_ds)} | val windows: {len(val_ds)} | test windows: {len(test_ds)}")
print(f"n_features: {X_train.shape[1]}")

## Modelo LSTM

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=32, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = out[:, -1, :]        # estado oculto del último paso temporal
        return self.head(last_hidden)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

n_features = X_train.shape[1]
model = LSTMRegressor(n_features=n_features, hidden_size=32, num_layers=1, dropout=0.3).to(device)
print(model)
print('Parámetros entrenables:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Entrenamiento (con early stopping sobre val loss)

Guardamos train loss y val loss por época. La brecha entre ambas curvas es el diagnóstico
principal de overfitting: si train loss sigue bajando mientras val loss sube, el modelo
está memorizando en vez de generalizar.

In [ ]:
import copy

N_EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4  # regularización L2, penaliza pesos grandes -> combate overfitting
PATIENCE = 8  # nº de épocas sin mejora en val loss antes de parar

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_samples = 0.0, 0
    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            n_samples += xb.size(0)
    return total_loss / n_samples

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_state = None
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader, model, criterion, optimizer)
    val_loss = run_epoch(val_loader, model, criterion, optimizer=None)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    marker = " *" if improved else ""
    print(f"Epoch {epoch:3d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping en epoch {epoch} (sin mejora en {PATIENCE} épocas).")
        break

# Restaurar los mejores pesos (por val loss), no los últimos -> evita quedarnos con un modelo sobreajustado
model.load_state_dict(best_state)
print(f"\nMejor val_loss: {best_val_loss:.5f}")

In [ ]:
# Curva de aprendizaje: train vs val loss por época
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train loss', linewidth=1.8)
plt.plot(val_losses, label='Val loss', linewidth=1.8)
best_epoch = int(np.argmin(val_losses))
plt.axvline(best_epoch, color='gray', linestyle='--', alpha=0.6, label=f'Mejor época ({best_epoch+1})')
plt.xlabel('Epoch')
plt.ylabel('MSE (escala estandarizada)')
plt.title('Curva de aprendizaje LSTM')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Predicciones y baseline naive

El target ahora es `TARGET_diff` (Δy). El baseline naive correspondiente ya no es
"y[t] = y[t-1]" sino **Δ=0** (predecir que no hay cambio) — es el equivalente exacto
del baseline de persistencia anterior, expresado en la serie diferenciada: predecir
Δ=0 en la serie diferenciada es idéntico a predecir y[t]=y[t-1] en la serie original.

In [ ]:
@torch.no_grad()
def predict(loader, model):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb).cpu().numpy()
        preds.append(out)
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

# Predicciones LSTM (en escala estandarizada, sobre TARGET_diff) para cada split.
# OJO: para train usamos 'train_eval_loader' (shuffle=False), no 'train_loader'
# (shuffle=True) -- de lo contrario las predicciones saldrían desordenadas y no
# se podrían alinear con el baseline naive ni con los timestamps.
train_pred_s, train_true_s = predict(train_eval_loader, model)
val_pred_s,   val_true_s   = predict(val_loader, model)
test_pred_s,  test_true_s  = predict(test_loader, model)

# Baseline naive para la serie DIFERENCIADA: predecir Δ=0 (sin cambio), que en la
# escala estandarizada de MinMaxScaler corresponde al valor de 0 mapeado, es decir
# scaler_y.transform([[0.0]]). Esto es distinto del baseline anterior "y[t-1]"
# porque ahora el target ya es un incremento, no un nivel.
zero_diff_scaled = scaler_y.transform([[0.0]])[0, 0]

def naive_baseline_diff(y_scaled, seq_len):
    y_hat = np.full_like(y_scaled[seq_len:], zero_diff_scaled)
    y_true = y_scaled[seq_len:]
    return y_hat, y_true

train_naive_pred_s, train_naive_true_s = naive_baseline_diff(y_train, SEQ_LEN)
val_naive_pred_s,   val_naive_true_s   = naive_baseline_diff(y_val,   SEQ_LEN)
test_naive_pred_s,  test_naive_true_s  = naive_baseline_diff(y_test,  SEQ_LEN)

# Sanity check: los "true" del naive deben coincidir exactamente con los del LSTM
assert np.allclose(train_naive_true_s, train_true_s)
assert np.allclose(val_naive_true_s, val_true_s)
assert np.allclose(test_naive_true_s, test_true_s)

# Vuelta a la escala original (Δ°C) para que las métricas sean interpretables
def inv(y_scaled):
    return scaler_y.inverse_transform(y_scaled.reshape(-1, 1)).ravel()

results = {
    'train': {
        'y_true': inv(train_true_s), 'lstm': inv(train_pred_s), 'naive': inv(train_naive_pred_s)
    },
    'val': {
        'y_true': inv(val_true_s), 'lstm': inv(val_pred_s), 'naive': inv(val_naive_pred_s)
    },
    'test': {
        'y_true': inv(test_true_s), 'lstm': inv(test_pred_s), 'naive': inv(test_naive_pred_s)
    },
}
print('Predicciones generadas para train/val/test, LSTM y naive (escala: Δ°C).')

## Métricas: LSTM vs naive, train / val / test

- **RMSE / MAE**: error absoluto en unidades del target.
- **MAPE**: error relativo (%), fácil de comunicar pero inestable si `y_true` se acerca a 0.
- **R²**: proporción de varianza explicada (1 = perfecto, 0 = tan bueno como predecir la media).
- **Skill score** = `1 - RMSE_lstm / RMSE_naive`: > 0 significa que el LSTM le gana a la persistencia;
  ≤ 0 significa que el modelo no aporta nada frente al baseline trivial.

Overfitting se lee comparando la fila `train` contra `val`/`test` de la **misma** métrica y modelo:
un RMSE de train mucho menor que el de val/test indica sobreajuste.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100

def compute_metrics(y_true, y_pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE (%)': mape(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

rows = []
for split in ['train', 'val', 'test']:
    y_true = results[split]['y_true']
    for model_name in ['lstm', 'naive']:
        m = compute_metrics(y_true, results[split][model_name])
        m['split'] = split
        m['model'] = model_name
        rows.append(m)

metrics_df = pd.DataFrame(rows).set_index(['split', 'model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']]
metrics_df = metrics_df.round(4)
print(metrics_df.to_string())

# Skill score: mejora relativa del LSTM sobre naive (RMSE), por split
print("\nSkill score (1 - RMSE_lstm / RMSE_naive) por split:")
for split in ['train', 'val', 'test']:
    rmse_lstm = metrics_df.loc[(split, 'lstm'), 'RMSE']
    rmse_naive = metrics_df.loc[(split, 'naive'), 'RMSE']
    skill = 1 - rmse_lstm / rmse_naive
    veredicto = "LSTM gana" if skill > 0 else "naive gana o empatan"
    print(f"  {split:5s}: skill={skill:+.4f}  ({veredicto})")

# Diagnóstico rápido de overfitting: ratio de RMSE test/train para el LSTM
rmse_ratio = metrics_df.loc[('test', 'lstm'), 'RMSE'] / metrics_df.loc[('train', 'lstm'), 'RMSE']
print(f"\nRatio RMSE test/train (LSTM): {rmse_ratio:.3f}  "
      f"({'posible overfitting' if rmse_ratio > 1.3 else 'generalización razonable'})")

## Predicción vs real (test) y residuos

Las métricas de arriba están en escala Δ°C (lo que el modelo predice directamente).
Para las gráficas, reconstruimos el nivel absoluto (°C) acumulando las diferencias
predichas a partir del último valor real conocido (`y_level[t] = y_level[t-1] + Δŷ[t]`).
Esta reconstrucción es útil para intuición visual, pero el error se acumula paso a
paso (es una suma acumulada) — cuanto más lejos del punto de partida, más se puede
alejar la reconstrucción del valor real incluso si cada Δ individual es razonable.
No se debe usar la reconstrucción para las métricas cuantitativas, solo para inspección visual.

Un buen modelo debe tener residuos (en Δ°C) centrados en 0, sin patrón temporal y con
varianza similar a lo largo del tiempo.

In [ ]:
# Timestamps correspondientes al test set, alineados con el offset SEQ_LEN
test_index = df.index[val_end:]
test_timestamps = test_index[SEQ_LEN:]

# Reconstrucción a nivel absoluto (°C) vía suma acumulada, partiendo del último
# nivel real conocido justo antes del test set (df[TARGET] en esa posición).
last_known_level = df[TARGET].iloc[val_end + SEQ_LEN - 1]

def reconstruct_level(diffs, start_level):
    return start_level + np.cumsum(diffs)

level_true_test = df[TARGET].iloc[val_end + SEQ_LEN:].values
level_lstm_test = reconstruct_level(results['test']['lstm'], last_known_level)
level_naive_test = reconstruct_level(results['test']['naive'], last_known_level)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 1. Nivel reconstruido vs real
axes[0].plot(test_timestamps, level_true_test, label='Real', color='black', linewidth=1.2)
axes[0].plot(test_timestamps, level_lstm_test, label='LSTM (reconstruido)', color='#1f77b4', linewidth=1.0, alpha=0.85)
axes[0].plot(test_timestamps, level_naive_test, label='Naive (reconstruido)', color='#ff7f0e', linewidth=1.0, alpha=0.7)
axes[0].set_title(f'Test set (nivel reconstruido): {TARGET}', loc='left', fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Residuos LSTM (escala Δ°C, la escala real de la métrica)
res_lstm = results['test']['y_true'] - results['test']['lstm']
axes[1].plot(test_timestamps, res_lstm, color='#1f77b4', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title(f'Residuos LSTM en Δ°C (media={res_lstm.mean():.4f}, std={res_lstm.std():.4f})', loc='left', fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

# 3. Residuos naive (escala Δ°C)
res_naive = results['test']['y_true'] - results['test']['naive']
axes[2].plot(test_timestamps, res_naive, color='#ff7f0e', linewidth=0.8)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title(f'Residuos Naive en Δ°C (media={res_naive.mean():.4f}, std={res_naive.std():.4f})', loc='left', fontweight='bold')
axes[2].grid(True, linestyle='--', alpha=0.5)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Predicción vs real en escala Δ°C (sin reconstrucción)

La gráfica anterior (nivel reconstruido) exagera visualmente el error del LSTM: un
sesgo medio pequeño en Δ°C se acumula paso a paso en la suma acumulada y termina
pareciendo un desvío enorme, aunque el error por paso sea comparable al de naive
(ver std de los residuos, prácticamente idéntico). Esta gráfica compara las
predicciones directamente en la escala en la que se entrena y se mide el modelo
(Δ°C), sin acumulación — es la comparación justa.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(test_timestamps, results['test']['y_true'], label='Real (Δ°C)', color='black', linewidth=1.0, alpha=0.8)
ax.plot(test_timestamps, results['test']['lstm'], label='LSTM (Δ°C)', color='#1f77b4', linewidth=0.9, alpha=0.8)
ax.plot(test_timestamps, results['test']['naive'], label='Naive (Δ°C)', color='#ff7f0e', linewidth=0.9, alpha=0.6)

ax.set_title(f'Test set — predicción directa en Δ°C (sin acumular): {TARGET}', loc='left', fontweight='bold')
ax.set_ylabel('Δ°C')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: predicho vs real (test), en escala Δ°C — la diagonal es la predicción perfecta
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

y_true_test = results['test']['y_true']
lims = [min(y_true_test.min(), results['test']['lstm'].min(), results['test']['naive'].min()),
        max(y_true_test.max(), results['test']['lstm'].max(), results['test']['naive'].max())]

for ax, model_name, color in zip(axes, ['lstm', 'naive'], ['#1f77b4', '#ff7f0e']):
    ax.scatter(y_true_test, results['test'][model_name], s=6, alpha=0.35, color=color)
    ax.plot(lims, lims, color='black', linewidth=1, linestyle='--', label='Predicción perfecta')
    ax.set_xlabel('Real (Δ°C)')
    ax.set_ylabel('Predicho (Δ°C)')
    ax.set_title(model_name.upper(), fontweight='bold')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

## Resumen de la investigación

**Target final:** `FirstStage.CombinerOperation.Temperature2.U.Actual`, diferenciado
(`TARGET_diff = Δy[t]`), `SEQ_LEN=10s`, LSTM de 32 unidades ocultas, 1 capa,
dropout 0.3, weight_decay 1e-4.

**Qué se descartó y por qué:**
- `Stage1.Output.Measurement8/14/6/13...`: todas las columnas `Stage1.Output.Measurement*`
  comparten un patrón de dropout hacia ~0 (el sensor solo "mide" de forma intermitente).
  En Measurement8 el dropout es un único burst al inicio de campaña (truncable); en el
  resto está disperso en bloques de ~4s por toda la serie — inviable para ventanas de
  60s y sin variabilidad real dentro del régimen activo (CV≈0.04).
- Nivel absoluto de `Temperature2`: train (53-115°C) y val/test (~102-108°C) caen en
  regímenes casi disjuntos — ningún modelo generaliza a un régimen que no vio en train.
  Confirmado (no fue overfitting ni falta de capacidad): reducir el modelo de 62.5k a
  10k parámetros empeoró el resultado en vez de mejorarlo.
- `LR=1e-2` y `MinMaxScaler` sobre un target con un burst de dropout: escalaba el rango
  útil (~20.9) a una porción mínima de [0,1], causando colapso del modelo hacia un
  valor de compromiso — visible como una nube plana en el scatter predicho-vs-real.
- **Remuestreo a 10s de resolución** (probado y descartado): la hipótesis era que a
  10s habría más dinámica real y menos ruido de medición que a 1s. Con el modelo
  original (32 unidades) el resultado fue catastrófico (test R²=-2365) por escasez de
  datos: el remuestreo deja solo ~1400 filas totales (~980 ventanas de train), 10x
  menos que a 1Hz, para un modelo con miles de parámetros. Incluso reduciendo
  drásticamente el modelo (8 unidades, dropout 0.4) para ajustar la capacidad a los
  datos disponibles, el resultado en test fue peor que a 1Hz (skill=-12.5% vs -1.6%).
  Con solo ~4h de campaña, no hay suficiente historia para que un split temporal
  70/15/15 a 10s de resolución deje ventanas suficientes para entrenar y validar bien.

**Qué funcionó:** diferenciar el target (`Δy[t] = y[t]-y[t-1]`) elimina el problema de
régimen no estacionario — la media pasa a ser ≈0 en los tres splits (antes eran
completamente distintas). Acortar `SEQ_LEN` de 60s a 10s (a 1Hz) ayudó más: el target
diferenciado cambia rápido, así que una ventana larga diluye la dinámica reciente
relevante con historia antigua poco útil.

**Resultado final (test, a 1Hz):** LSTM RMSE=0.331 Δ°C vs naive RMSE=0.326 Δ°C
(skill=-1.6%, prácticamente empate). El LSTM no logra superar la persistencia, pero
se queda muy cerca — resultado razonable y esperable: a resolución de 1s, el cambio de
esta variable de proceso es casi un paseo aleatorio de ruido, con poca estructura
predecible más allá de "el valor no cambia mucho de un segundo a otro". Superar la
persistencia en estas condiciones requeriría más señal (features adicionales,
posiblemente lags explícitos del propio target) o predecir a un horizonte más largo
donde haya más dinámica real que capturar — pero para eso hace falta más historia de
la que ofrece esta única campaña de ~4h.

# Extensión: predicción conjunta de las 3 temperaturas del combinador

En vez de un único target, predecimos a la vez `Temperature1`, `Temperature2` y
`Temperature3` con un solo LSTM (tronco compartido + cabeza de salida de 3 unidades).
Reutilizamos exactamente el mismo enfoque que funcionó arriba: diferenciar cada target
antes de entrenar.

**Por qué diferenciar las 3, no solo Temperature2:**
- `Temperature1`: mismo patrón que Temperature2 aunque más suave — train
  (media 106.0, std 4.0, incluye el outlier de 45.3) vs val/test (~115-116, std <1.6).
  Mismo desplazamiento de régimen.
- `Temperature2`: el caso ya analizado — desplazamiento de régimen fuerte
  (train ~76.5 vs val/test ~104.5).
- `Temperature3`: prácticamente estacionaria en las 3 particiones (media ≈80.0,
  std ≈0.1-0.13 en train/val/test) — no tiene el problema, pero diferenciarla no
  hace daño y mantiene el pipeline uniforme entre los 3 targets.

**Escalado y loss:** cada target se escala de forma independiente (un `MinMaxScaler`
por columna) antes de construir el tensor de 3 columnas — así el MSE conjunto no
queda dominado por el target de mayor varianza (Temperature2 tiene un std de diff
~10x mayor que Temperature3). No se añaden pesos manuales por target: al estar los
tres en la misma escala [0,1], una suma/media simple de MSE ya los pondera de forma
comparable.

In [ ]:
MULTI_TARGETS = [
    'FirstStage.CombinerOperation.Temperature1.U.Actual',
    'FirstStage.CombinerOperation.Temperature2.U.Actual',
    'FirstStage.CombinerOperation.Temperature3.C.Actual',
]
MULTI_TARGET_DIFFS = [f'{t}_diff' for t in MULTI_TARGETS]

for t, td in zip(MULTI_TARGETS, MULTI_TARGET_DIFFS):
    df[td] = df[t].diff()

# La primera fila ya se descartó en el bloque de un solo target (TARGET_diff),
# pero por claridad repetimos el filtrado: cualquier fila con algún diff nulo se cae.
df_multi = df.dropna(subset=MULTI_TARGET_DIFFS).copy()

print(f"Filas disponibles: {len(df_multi)}")
print(df_multi[MULTI_TARGET_DIFFS].describe())

## Selección de features por información mutua (no lineal)

La correlación de Pearson entre cada una de las 38 features y cada target diferenciado
resultó ser prácticamente nula en todos los casos (|r| < 0.02 siempre) — no hay
relación *lineal* detectable. Eso no descarta relaciones no lineales, que un LSTM sí
puede explotar en principio. Calculamos información mutua
(`sklearn.feature_selection.mutual_info_regression`) entre cada feature y cada uno de
los 3 targets diferenciados.

**Fuga de datos detectada y corregida:** la primera versión de este análisis calculó
la información mutua sobre `df_multi` completo (train+val+test, ~14k filas), no solo
sobre train. Eso significa que la selección de qué features "importan" estaba viendo
datos de val/test antes de que existiera el split — una fuga silenciosa, aunque esos
datos nunca entraran directamente al modelo. Al corregirlo (información mutua
calculada solo sobre el 70% de train, con el mismo corte `train_end_m` que usa el
split real):
- La lista de top features cambió de forma notable: `Machine1.ExitZoneTemperature` y
  `Machine3.MotorRPM` (antes arriba de la lista) salen del top-15; entran
  `Machine1.MotorRPM`, `Machine1.RawMaterial.Property3`, `Machine2.RawMaterial.Property2`.
- El resultado empeoró sustancialmente, sobre todo en val: el skill de Temperature1 en
  val pasó de -8.5% (con la fuga) a **-247%** (corregido); Temperature2 de -1.75% a
  **-176%**. En test la caída es más moderada pero también negativa: Temperature1
  -0.02%→-5.8%, Temperature2 -0.28%→-2.9%.

Esto confirma que la aparente "señal real" en Temperature1 que vimos en la primera
pasada era en parte un artefacto de la fuga: al forzar que la selección de features
sea honesta (solo con lo que el modelo vería en producción, es decir, solo train), esa
señal se debilita mucho. Es una lección metodológica más importante que el propio
resultado: cualquier paso de "elegir qué usar" (selección de features, tuning de
hiperparámetros, etc.) debe hacerse solo con datos de train, igual que el fit de los
scalers — de lo contrario el número final de test deja de ser una estimación honesta
de generalización.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Features candidatas: todas menos los niveles absolutos y sus diffs (igual que antes).
all_feature_cols = [c for c in df_multi.columns if c not in
                     (['time_stamp', TARGET, target_col] + MULTI_TARGETS + MULTI_TARGET_DIFFS)]

# OJO: la información mutua se calcula SOLO sobre train, nunca sobre todo df_multi.
# Si se calculara sobre el dataset completo (train+val+test), la selección de
# features estaría "viendo" datos de val/test antes de que exista el split -- una
# fuga de información silenciosa, aunque los datos de val/test nunca entren
# directamente al modelo. Usamos el mismo corte train_end_m que se usa más abajo
# para el split real.
n_multi = len(df_multi)
train_end_mi = int(n_multi * 0.70)
df_multi_train_only = df_multi.iloc[:train_end_mi]

X_for_mi = df_multi_train_only[all_feature_cols].to_numpy()

mi_per_target = {}
for t in MULTI_TARGETS:
    y_for_mi = df_multi_train_only[f'{t}_diff'].to_numpy()
    mi_per_target[t] = mutual_info_regression(X_for_mi, y_for_mi, random_state=0, n_neighbors=3)

mi_df = pd.DataFrame(mi_per_target, index=all_feature_cols)
mi_df['max_MI'] = mi_df.max(axis=1)
mi_df = mi_df.sort_values('max_MI', ascending=False)

N_TOP_FEATURES = 15
selected_features = mi_df.head(N_TOP_FEATURES).index.tolist()

print(f"Top {N_TOP_FEATURES} features por información mutua máxima (calculada solo en train):")
print(mi_df.head(N_TOP_FEATURES).round(5).to_string())
print(f"\nFeatures con MI=0 en los 3 targets (ruido puro): "
      f"{mi_df.index[(mi_df[MULTI_TARGETS] == 0).all(axis=1)].tolist()}")

In [ ]:
# Features: solo las top-15 por información mutua (en vez de todas las 38).
multi_feature_cols = selected_features

X_raw_m = df_multi[multi_feature_cols].values
Y_raw_m = df_multi[MULTI_TARGET_DIFFS].values  # (n, 3)

n_m = len(df_multi)
train_end_m = int(n_m * 0.70)
val_end_m = int(n_m * 0.85)

X_train_raw_m, Y_train_raw_m = X_raw_m[:train_end_m], Y_raw_m[:train_end_m]
X_val_raw_m,   Y_val_raw_m   = X_raw_m[train_end_m:val_end_m], Y_raw_m[train_end_m:val_end_m]
X_test_raw_m,  Y_test_raw_m  = X_raw_m[val_end_m:], Y_raw_m[val_end_m:]

# Escalado independiente por columna (features y cada uno de los 3 targets), fit solo en train.
scaler_X_m = MinMaxScaler()
X_train_m = scaler_X_m.fit_transform(X_train_raw_m)
X_val_m   = scaler_X_m.transform(X_val_raw_m)
X_test_m  = scaler_X_m.transform(X_test_raw_m)

# MinMaxScaler escala cada COLUMNA de forma independiente cuando la entrada tiene
# varias columnas, así que un único scaler ya escala los 3 targets por separado.
scaler_Y_m = MinMaxScaler()
Y_train_m = scaler_Y_m.fit_transform(Y_train_raw_m)
Y_val_m   = scaler_Y_m.transform(Y_val_raw_m)
Y_test_m  = scaler_Y_m.transform(Y_test_raw_m)

print(f"Train: {X_train_m.shape[0]} filas | Val: {X_val_m.shape[0]} | Test: {X_test_m.shape[0]}")
print(f"n_features: {X_train_m.shape[1]} | n_targets: {Y_train_m.shape[1]}")

In [ ]:
# Reutilizamos SequenceDataset (ya es genérica sobre la forma de 'y': funciona igual
# con un target que con varios, porque simplemente indexa y[idx+seq_len]).
train_ds_m = SequenceDataset(X_train_m, Y_train_m, SEQ_LEN)
val_ds_m   = SequenceDataset(X_val_m,   Y_val_m,   SEQ_LEN)
test_ds_m  = SequenceDataset(X_test_m,  Y_test_m,  SEQ_LEN)

train_loader_m = DataLoader(train_ds_m, batch_size=BATCH_SIZE, shuffle=True)
val_loader_m   = DataLoader(val_ds_m,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_m  = DataLoader(test_ds_m,  batch_size=BATCH_SIZE, shuffle=False)
train_eval_loader_m = DataLoader(train_ds_m, batch_size=BATCH_SIZE, shuffle=False)

print(f"train windows: {len(train_ds_m)} | val windows: {len(val_ds_m)} | test windows: {len(test_ds_m)}")

In [ ]:
class LSTMMultiRegressor(nn.Module):
    """Igual que LSTMRegressor de antes, pero la última capa tiene n_targets salidas
    en vez de 1. El tronco (LSTM) se comparte entre los 3 targets."""
    def __init__(self, n_features, n_targets, hidden_size=32, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, n_targets),
        )

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.head(last_hidden)

n_features_m = X_train_m.shape[1]
n_targets_m = Y_train_m.shape[1]
model_m = LSTMMultiRegressor(n_features=n_features_m, n_targets=n_targets_m,
                              hidden_size=32, num_layers=1, dropout=0.3).to(device)
print(model_m)
print('Parámetros entrenables:', sum(p.numel() for p in model_m.parameters() if p.requires_grad))

In [ ]:
# Reutilizamos run_epoch: nn.MSELoss() por defecto promedia sobre todos los elementos
# del tensor de salida, así que con (batch, 3) simplemente promedia también entre targets.
# Esto es lo que discutíamos como "suma/media simple, sin pesos manuales por target".
optimizer_m = torch.optim.Adam(model_m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_m = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_m, mode='min', factor=0.5, patience=3)

train_losses_m, val_losses_m = [], []
best_val_loss_m = float('inf')
best_state_m = None
epochs_no_improve_m = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader_m, model_m, criterion, optimizer_m)
    val_loss = run_epoch(val_loader_m, model_m, criterion, optimizer=None)
    scheduler_m.step(val_loss)

    train_losses_m.append(train_loss)
    val_losses_m.append(val_loss)

    improved = val_loss < best_val_loss_m
    if improved:
        best_val_loss_m = val_loss
        best_state_m = copy.deepcopy(model_m.state_dict())
        epochs_no_improve_m = 0
    else:
        epochs_no_improve_m += 1

    marker = " *" if improved else ""
    print(f"Epoch {epoch:3d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}{marker}")

    if epochs_no_improve_m >= PATIENCE:
        print(f"Early stopping en epoch {epoch} (sin mejora en {PATIENCE} épocas).")
        break

model_m.load_state_dict(best_state_m)
print(f"\nMejor val_loss: {best_val_loss_m:.5f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses_m, label='Train loss', linewidth=1.8)
plt.plot(val_losses_m, label='Val loss', linewidth=1.8)
best_epoch_m = int(np.argmin(val_losses_m))
plt.axvline(best_epoch_m, color='gray', linestyle='--', alpha=0.6, label=f'Mejor época ({best_epoch_m+1})')
plt.xlabel('Epoch')
plt.ylabel('MSE (escala estandarizada, promedio de 3 targets)')
plt.title('Curva de aprendizaje LSTM multi-target')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Predicciones, baseline naive y métricas por target

El baseline naive sigue siendo Δ=0 (sin cambio) para cada uno de los 3 targets por
separado. Reportamos RMSE/MAE/R² y el skill score individualmente por target, porque
no tiene sentido promediarlos entre sí: cada temperatura tiene su propia escala y su
propio grado de dificultad (recordemos que Temperature3 es casi estacionaria y
Temperature1/2 tenían el problema de régimen).

In [ ]:
@torch.no_grad()
def predict_multi(loader, model):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb).cpu().numpy()
        preds.append(out)
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)  # (n, 3) cada uno

train_pred_sm, train_true_sm = predict_multi(train_eval_loader_m, model_m)
val_pred_sm,   val_true_sm   = predict_multi(val_loader_m, model_m)
test_pred_sm,  test_true_sm  = predict_multi(test_loader_m, model_m)

# Baseline naive multi-target: Δ=0 (columna a columna) en escala estandarizada.
zero_diff_scaled_m = scaler_Y_m.transform(np.zeros((1, n_targets_m)))[0]

def naive_baseline_multi(Y_scaled, seq_len):
    y_hat = np.tile(zero_diff_scaled_m, (len(Y_scaled) - seq_len, 1))
    y_true = Y_scaled[seq_len:]
    return y_hat, y_true

train_naive_pred_sm, train_naive_true_sm = naive_baseline_multi(Y_train_m, SEQ_LEN)
val_naive_pred_sm,   val_naive_true_sm   = naive_baseline_multi(Y_val_m,   SEQ_LEN)
test_naive_pred_sm,  test_naive_true_sm  = naive_baseline_multi(Y_test_m,  SEQ_LEN)

assert np.allclose(train_naive_true_sm, train_true_sm)
assert np.allclose(val_naive_true_sm, val_true_sm)
assert np.allclose(test_naive_true_sm, test_true_sm)

def inv_multi(Y_scaled):
    return scaler_Y_m.inverse_transform(Y_scaled)

results_m = {
    'train': {'y_true': inv_multi(train_true_sm), 'lstm': inv_multi(train_pred_sm), 'naive': inv_multi(train_naive_pred_sm)},
    'val':   {'y_true': inv_multi(val_true_sm),   'lstm': inv_multi(val_pred_sm),   'naive': inv_multi(val_naive_pred_sm)},
    'test':  {'y_true': inv_multi(test_true_sm),  'lstm': inv_multi(test_pred_sm),  'naive': inv_multi(test_naive_pred_sm)},
}

# Métricas por target y por split
rows_m = []
for split in ['train', 'val', 'test']:
    for ti, target_name in enumerate(MULTI_TARGETS):
        y_true = results_m[split]['y_true'][:, ti]
        for model_name in ['lstm', 'naive']:
            y_pred = results_m[split][model_name][:, ti]
            m = compute_metrics(y_true, y_pred)
            m['split'] = split
            m['target'] = target_name.replace('FirstStage.CombinerOperation.', '')
            m['model'] = model_name
            rows_m.append(m)

metrics_df_m = pd.DataFrame(rows_m).set_index(['split', 'target', 'model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']]
metrics_df_m = metrics_df_m.round(4)
print(metrics_df_m.to_string())

print("\nSkill score (1 - RMSE_lstm / RMSE_naive) por target y split:")
for split in ['train', 'val', 'test']:
    for target_name in MULTI_TARGETS:
        tname = target_name.replace('FirstStage.CombinerOperation.', '')
        rmse_lstm = metrics_df_m.loc[(split, tname, 'lstm'), 'RMSE']
        rmse_naive = metrics_df_m.loc[(split, tname, 'naive'), 'RMSE']
        skill = 1 - rmse_lstm / rmse_naive
        print(f"  {split:5s} | {tname:15s}: skill={skill:+.4f}")

In [ ]:
# Timestamps del test set multi-target (mismo offset SEQ_LEN, sobre df_multi)
test_index_m = df_multi.index[val_end_m:]
test_timestamps_m = test_index_m[SEQ_LEN:]

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

for ti, (ax, target_name) in enumerate(zip(axes, MULTI_TARGETS)):
    tname = target_name.replace('FirstStage.CombinerOperation.', '')
    ax.plot(test_timestamps_m, results_m['test']['y_true'][:, ti], label='Real', color='black', linewidth=1.0, alpha=0.8)
    ax.plot(test_timestamps_m, results_m['test']['lstm'][:, ti], label='LSTM', color='#1f77b4', linewidth=0.9, alpha=0.8)
    ax.plot(test_timestamps_m, results_m['test']['naive'][:, ti], label='Naive', color='#ff7f0e', linewidth=0.9, alpha=0.6)
    ax.set_title(f'Test set — {tname} (Δ°C, sin acumular)', loc='left', fontweight='bold')
    ax.set_ylabel('Δ°C')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Conclusión de la extensión multi-target

El LSTM conjunto (con las 38 features, sin selección) entrena de forma estable (curva
de aprendizaje suave, sin divergencia) y empata con el baseline naive en las 3
temperaturas por separado:

| Target | Skill score (test, 38 features) |
|---|---|
| Temperature1 | -0.03% |
| Temperature2 | -0.64% |
| Temperature3 | 0.00% |

Mismo patrón que en el caso de un solo target: a resolución de 1s, el cambio de estas
variables es casi ruido puro, y ni compartir el tronco del LSTM entre las 3
temperaturas (que podría aportar señal cruzada si estuvieran correlacionadas en su
dinámica de corto plazo) le da al modelo ventaja sobre "no cambia nada".

**Después de probar selección de features** (información mutua, calculada
correctamente solo sobre train — ver sección siguiente), el resultado con el top-15 de
features fue *peor*, no mejor: skill de test Temperature1 -5.8%, Temperature2 -2.9%,
Temperature3 ≈0%. Esto refuerza la conclusión: el problema no es que algunas features
"metan ruido" y opaquen a las buenas — es que, evaluado honestamente (sin fuga de
datos), ninguna de las 38 features tiene relación suficiente con el cambio de estas
temperaturas segundo a segundo como para que el LSTM saque ventaja sobre la
persistencia, ni usando todas las features ni restringiendo a un subconjunto.

## Resumen: ¿algunas señales estaban perjudicando al LSTM?

**No.** Se comprobó de dos formas independientes:

1. **Correlación de Pearson** entre las 38 features y cada target diferenciado:
   todas por debajo de |r|=0.02 — sin relación lineal detectable en ninguna.
2. **Información mutua** (captura relaciones no lineales, que un LSTM sí puede
   explotar), calculada correctamente solo sobre train: unas pocas features muestran
   algo de estructura (máximo MI≈0.027, `Machine1.MotorRPM` con Temperature2), pero
   es una señal débil, y restringir el modelo a esas top-15 features dio resultados
   *peores* que usar las 38, no mejores.

La primera versión de este análisis (información mutua calculada sobre todo el
dataset, no solo train) sugería que sí había señal recuperable en Temperature1 — pero
era en parte un artefacto de una fuga de datos (selección de features "viendo" val/test
antes del split). Al corregirla, esa señal se debilitó sustancialmente.

**Conclusión:** el techo de rendimiento del LSTM en este problema (empatar, no superar,
la persistencia) no se debe a que unas señales concretas estén "contaminando" el
aprendizaje. Ninguna de las 38 variables de proceso disponibles contiene información
suficiente — lineal o no lineal — sobre cómo cambian estas temperaturas segundo a
segundo. El cambio en `Δy` a 1Hz se comporta, para estos datos, muy cerca de ruido
puro respecto a las variables medidas.

# Extensión: ¿tienen las features de entrada su propio desplazamiento de régimen?

Buena pregunta que no habíamos comprobado: igual que `Temperature1`/`Temperature2`
en nivel absoluto tenían un desplazamiento de régimen fuerte entre train y val/test,
¿pasa lo mismo con alguna de las 38 features de entrada?

**Sí.** Comprobando el hueco entre la media de train y la de val/test (en unidades de
desviación estándar de la señal completa), varias features de `Machine1` muestran
huecos >2σ (`Machine1.RawMaterial.Property2/3/4`, `Machine1.MaterialPressure`,
`Machine1.MotorRPM`). Y más llamativo aún: varias columnas `RawMaterial.Property*`
(sobre todo de `Machine2` y `Machine3`) muestran un **desplazamiento de régimen incluso
dentro del propio train** (hasta 2.26σ entre los tres tercios cronológicos de train) —
consistente con que estas columnas son casi variables categóricas/de lote (cambian a
saltos cuando cambia el lote de materia prima), no señales continuas.

**Prueba:** diferenciamos las 38 features de entrada (igual que ya hacíamos con los
targets) y recalculamos correlación e información mutua (siempre solo sobre train, sin
la fuga de datos que corregimos antes):

| | Correlación máx. media | Correlación máx. absoluta | MI máx. media | MI máx. absoluta |
|---|---|---|---|---|
| Features en nivel (crudo) | 0.0068 | 0.0177 | 0.0096 | 0.0225 |
| Features diferenciadas | 0.0105 | **0.0377** | 0.0078 | 0.0188 |

Diferenciar las features de entrada casi duplica la mejor correlación lineal
encontrada (0.018→0.038), aunque empeora ligeramente la información mutua máxima
(0.023→0.019) — probablemente porque diferenciar amplifica el ruido de alta frecuencia
más de lo que ayuda a un estimador no lineal tipo kNN, mientras que sí ayuda a una
métrica lineal si la relación real es entre *tasas de cambio* (Δfeature vs Δtarget) en
vez de entre niveles. En cualquier caso, ambos techos siguen siendo demasiado bajos
para que un modelo saque partido de forma fiable.

Para comprobarlo a nivel de modelo, reentrenamos el LSTM multi-target usando las 38
features **diferenciadas** en vez de en nivel, y comparamos el skill score contra el
resultado ya obtenido con las features en nivel.

In [ ]:
# Diferenciamos también las features de entrada (all_feature_cols ya excluye niveles
# y diffs de los 3 targets).
feature_diff_cols = [f'{c}_diff' for c in all_feature_cols]
for c, cd in zip(all_feature_cols, feature_diff_cols):
    df_multi[cd] = df_multi[c].diff()

# La primera fila queda NaN (sin t-1 anterior) -- se descarta igual que con los targets.
df_multi_diffed = df_multi.dropna(subset=feature_diff_cols).copy()

print(f"Filas antes: {len(df_multi)} | después de diferenciar features: {len(df_multi_diffed)}")
print(df_multi_diffed[feature_diff_cols].describe().T[['mean', 'std']].round(4).to_string())

In [ ]:
# Split/escalado igual que en los bloques anteriores, pero usando las features
# diferenciadas (feature_diff_cols) en vez de en nivel.
X_raw_d = df_multi_diffed[feature_diff_cols].values
Y_raw_d = df_multi_diffed[MULTI_TARGET_DIFFS].values

n_d = len(df_multi_diffed)
train_end_d = int(n_d * 0.70)
val_end_d = int(n_d * 0.85)

X_train_raw_d, Y_train_raw_d = X_raw_d[:train_end_d], Y_raw_d[:train_end_d]
X_val_raw_d,   Y_val_raw_d   = X_raw_d[train_end_d:val_end_d], Y_raw_d[train_end_d:val_end_d]
X_test_raw_d,  Y_test_raw_d  = X_raw_d[val_end_d:], Y_raw_d[val_end_d:]

scaler_X_d = MinMaxScaler()
X_train_d = scaler_X_d.fit_transform(X_train_raw_d)
X_val_d   = scaler_X_d.transform(X_val_raw_d)
X_test_d  = scaler_X_d.transform(X_test_raw_d)

scaler_Y_d = MinMaxScaler()
Y_train_d = scaler_Y_d.fit_transform(Y_train_raw_d)
Y_val_d   = scaler_Y_d.transform(Y_val_raw_d)
Y_test_d  = scaler_Y_d.transform(Y_test_raw_d)

print(f"Train: {X_train_d.shape[0]} filas | Val: {X_val_d.shape[0]} | Test: {X_test_d.shape[0]}")
print(f"n_features: {X_train_d.shape[1]} | n_targets: {Y_train_d.shape[1]}")

In [ ]:
train_ds_d = SequenceDataset(X_train_d, Y_train_d, SEQ_LEN)
val_ds_d   = SequenceDataset(X_val_d,   Y_val_d,   SEQ_LEN)
test_ds_d  = SequenceDataset(X_test_d,  Y_test_d,  SEQ_LEN)

train_loader_d = DataLoader(train_ds_d, batch_size=BATCH_SIZE, shuffle=True)
val_loader_d   = DataLoader(val_ds_d,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_d  = DataLoader(test_ds_d,  batch_size=BATCH_SIZE, shuffle=False)
train_eval_loader_d = DataLoader(train_ds_d, batch_size=BATCH_SIZE, shuffle=False)

print(f"train windows: {len(train_ds_d)} | val windows: {len(val_ds_d)} | test windows: {len(test_ds_d)}")

n_features_d = X_train_d.shape[1]
n_targets_d = Y_train_d.shape[1]
model_d = LSTMMultiRegressor(n_features=n_features_d, n_targets=n_targets_d,
                              hidden_size=32, num_layers=1, dropout=0.3).to(device)
print('Parámetros entrenables:', sum(p.numel() for p in model_d.parameters() if p.requires_grad))

In [ ]:
optimizer_d = torch.optim.Adam(model_d.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_d = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_d, mode='min', factor=0.5, patience=3)

train_losses_d, val_losses_d = [], []
best_val_loss_d = float('inf')
best_state_d = None
epochs_no_improve_d = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader_d, model_d, criterion, optimizer_d)
    val_loss = run_epoch(val_loader_d, model_d, criterion, optimizer=None)
    scheduler_d.step(val_loss)

    train_losses_d.append(train_loss)
    val_losses_d.append(val_loss)

    improved = val_loss < best_val_loss_d
    if improved:
        best_val_loss_d = val_loss
        best_state_d = copy.deepcopy(model_d.state_dict())
        epochs_no_improve_d = 0
    else:
        epochs_no_improve_d += 1

    marker = " *" if improved else ""
    print(f"Epoch {epoch:3d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}{marker}")

    if epochs_no_improve_d >= PATIENCE:
        print(f"Early stopping en epoch {epoch} (sin mejora en {PATIENCE} épocas).")
        break

model_d.load_state_dict(best_state_d)
print(f"\nMejor val_loss: {best_val_loss_d:.5f}")

In [ ]:
train_pred_sd, train_true_sd = predict_multi(train_eval_loader_d, model_d)
val_pred_sd,   val_true_sd   = predict_multi(val_loader_d, model_d)
test_pred_sd,  test_true_sd  = predict_multi(test_loader_d, model_d)

zero_diff_scaled_d = scaler_Y_d.transform(np.zeros((1, n_targets_d)))[0]

def naive_baseline_multi_d(Y_scaled, seq_len):
    y_hat = np.tile(zero_diff_scaled_d, (len(Y_scaled) - seq_len, 1))
    y_true = Y_scaled[seq_len:]
    return y_hat, y_true

train_naive_pred_sd, train_naive_true_sd = naive_baseline_multi_d(Y_train_d, SEQ_LEN)
val_naive_pred_sd,   val_naive_true_sd   = naive_baseline_multi_d(Y_val_d,   SEQ_LEN)
test_naive_pred_sd,  test_naive_true_sd  = naive_baseline_multi_d(Y_test_d,  SEQ_LEN)

assert np.allclose(train_naive_true_sd, train_true_sd)
assert np.allclose(val_naive_true_sd, val_true_sd)
assert np.allclose(test_naive_true_sd, test_true_sd)

def inv_multi_d(Y_scaled):
    return scaler_Y_d.inverse_transform(Y_scaled)

results_d = {
    'train': {'y_true': inv_multi_d(train_true_sd), 'lstm': inv_multi_d(train_pred_sd), 'naive': inv_multi_d(train_naive_pred_sd)},
    'val':   {'y_true': inv_multi_d(val_true_sd),   'lstm': inv_multi_d(val_pred_sd),   'naive': inv_multi_d(val_naive_pred_sd)},
    'test':  {'y_true': inv_multi_d(test_true_sd),  'lstm': inv_multi_d(test_pred_sd),  'naive': inv_multi_d(test_naive_pred_sd)},
}

rows_d = []
for split in ['train', 'val', 'test']:
    for ti, target_name in enumerate(MULTI_TARGETS):
        y_true = results_d[split]['y_true'][:, ti]
        for model_name in ['lstm', 'naive']:
            y_pred = results_d[split][model_name][:, ti]
            m = compute_metrics(y_true, y_pred)
            m['split'] = split
            m['target'] = target_name.replace('FirstStage.CombinerOperation.', '')
            m['model'] = model_name
            rows_d.append(m)

metrics_df_d = pd.DataFrame(rows_d).set_index(['split', 'target', 'model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']]
metrics_df_d = metrics_df_d.round(4)
print(metrics_df_d.to_string())

print("\nSkill score (features diferenciadas) por target y split:")
for split in ['train', 'val', 'test']:
    for target_name in MULTI_TARGETS:
        tname = target_name.replace('FirstStage.CombinerOperation.', '')
        rmse_lstm = metrics_df_d.loc[(split, tname, 'lstm'), 'RMSE']
        rmse_naive = metrics_df_d.loc[(split, tname, 'naive'), 'RMSE']
        skill = 1 - rmse_lstm / rmse_naive
        print(f"  {split:5s} | {tname:15s}: skill={skill:+.4f}")

## Conclusión: features de entrada diferenciadas vs en nivel

| Target | Features en nivel (skill test) | Features diferenciadas (skill test) |
|---|---|---|
| Temperature1 | -0.03% | -0.34% |
| Temperature2 | -0.64% | -0.71% |
| Temperature3 | 0.00% | 0.00% |

Diferenciar las 38 features de entrada **empeora ligeramente** el resultado en test en
vez de mejorarlo, en los tres targets. Esto es coherente con lo que ya habíamos visto
en el chequeo de información mutua (la métrica no lineal, más relevante para lo que un
LSTM puede aprovechar): diferenciar las features bajó la información mutua máxima
(0.0225→0.0188) aunque subiera la correlación lineal (0.0177→0.0377). El LSTM sigue el
patrón de la información mutua, no el de la correlación — tiene sentido, porque un
LSTM puede explotar relaciones no lineales que la correlación de Pearson no detecta.

**Conclusión final de toda esta línea de investigación** (arranca en "¿algunas señales
están perjudicando al LSTM?"): no. Se probó (a) seleccionar solo las features más
informativas, (b) suavizar y recortar outliers de las features, y (c) diferenciar las
features para eliminar sus propios desplazamientos de régimen — ninguna de las tres
intervenciones supera al modelo base con las 38 features en nivel absoluto (que ya
empataba, sin ganar, a la persistencia). El modelo con las 38 features en nivel sigue
siendo la configuración más simple y con mejor resultado de todas las probadas.

# Extensión: ¿el ruido en las features es lo que empeora al diferenciarlas?

Buena hipótesis: diferenciar amplifica ruido de alta frecuencia (varianza del ruido
se duplica aproximadamente al restar dos muestras independientes), mientras que la
señal real (que cambia despacio) apenas cambia al diferenciar. Si una feature es
mayormente ruido, diferenciarla puede empeorar su relación con el target en vez de
mejorarla.

**Comprobación 1 — ¿hay features dominadas por ruido?** Sí. Estimando "ruido" como el
residuo entre cada feature y una versión suavizada (media móvil de 15s) de sí misma,
solo train:

| Feature | Fracción de varianza que es ruido | Varianza tras diferenciar / varianza en nivel |
|---|---|---|
| `Machine2.MotorRPM` | 95.9% | ×1.77 (amplifica) |
| `Machine3.RawMaterialFeederParameter` | 83.9% | ×1.74 (amplifica) |
| `Machine2.RawMaterialFeederParameter` | 83.6% | ×1.74 (amplifica) |
| `Machine2.MaterialPressure` | 52.8% | ×0.80 |
| `Machine2.MotorAmperage` | 26.2% | ×0.43 |
| *(resto de las 38 features)* | <20%, mayoría <1% | ×0.0001–0.02 (se reduce mucho) |

Estas 5 features sí encajan con la hipótesis: mayoritariamente ruido, y diferenciar
aumenta su varianza. El resto de features (la mayoría) son al revés — su varianza en
nivel está dominada por saltos de régimen/lote (visto en la sección anterior), así que
diferenciar les *reduce* drásticamente la varianza, no la amplifica.

**Comprobación 2 — ¿ayuda suavizar (quitar el ruido) antes de diferenciar, aunque sea
solo en esas 5 features?** No, tampoco. Suavizando solo las 5 features con más ruido y
diferenciando después: la información mutua media con los targets bajó ligeramente
(0.0078→0.0074) y, sorprendentemente, **3 de las 5 features empeoraron** tras quitarles
el ruido (`Machine2.MotorRPM`: 0.0188→0.0102; `Machine3.RawMaterialFeederParameter`:
0.0078→0.0000; `Machine2.MotorAmperage`: 0.0105→0.0083). Solo 2 mejoraron, y poco.

**Conclusión:** el mecanismo de amplificación de ruido es real y medible (algunas
features sí son mayoritariamente ruido, y diferenciarlas sí incrementa su varianza),
pero corregirlo no recupera señal predictiva — ni siquiera en las features donde el
ruido es el problema dominante. Esto refuerza la conclusión de fondo: el techo bajo de
información mutua (~0.02) no es un problema de ruido que se pueda limpiar, es que
—incluso limpio— no hay suficiente relación real entre estas variables de proceso y el
cambio segundo a segundo de las temperaturas del combinador.

# Extensión: remuestreo a 60 segundos

Idea: quizás la relación entre las features de proceso y las temperaturas es más
visible a escala de minutos que a escala de segundo a segundo (donde parece casi
ruido puro). Remuestreamos a 1 muestra cada 60s (media por bucket) y repetimos el
análisis de señal (información mutua) y además probamos causalidad de Granger.

**Nota técnica:** `np.corrcoef`, `np.linalg.lstsq` e incluso la multiplicación de
matrices `@` están bloqueados en este entorno por la misma política de control de
aplicaciones que ya bloqueaba matplotlib/pywin32 en sesiones anteriores. Para calcular
Granger (que requiere ajustar regresiones OLS) implementamos un solver manual por
eliminación de Gauss-Jordan usando solo numpy elemento a elemento, sin `np.linalg` ni
`@`.

**Resultado y por qué hay que desconfiar de él a primera vista:** remuestrear a 60s
reduce el dataset de 14,088 a solo 236 filas (164 de train tras el split y el diff).
La información mutua sube mucho en apariencia (media 0.096, máximo 0.19, frente a
0.0096/0.0225 a 1s) — pero **una comparación directa de esos números es inválida**: el
estimador de información mutua basado en vecinos más cercanos (kNN) está sesgado al
alza con muestras pequeñas, independientemente de si existe relación real. No se puede
comparar MI calculada con 164 filas contra MI calculada con ~9800 filas.

**Test de permutación (la comprobación correcta):** barajamos el target 20 veces (rompe
cualquier relación real, mantiene la distribución marginal) y recalculamos MI y Granger
sobre los mismos datos barajados.
- MI real (0.184) queda por encima de las 20 permutaciones (máximo barajado: 0.128) —
  hay más estructura que puro ruido, pero **incluso los datos barajados dan MI≈0.096
  de media**, ~10 veces la MI real que encontramos a 1s con tamaño de muestra correcto.
  Esto confirma que el estimador está inflado por el tamaño de muestra tan pequeño, no
  solo que hay señal real.
- Granger (lag=2) para `Machine2/3.RawMaterialFeederParameter` sí supera claramente al
  barajado (F=5.3/6.2 reales vs máximo barajado ~3.5) — esto parece señal real, no
  solo sesgo de muestra pequeña. Pero estas son justo las dos features "de lote" que ya
  identificamos con desplazamientos de régimen dentro de train — es más probable que
  reflejen coincidencia temporal de cambios de régimen (un cambio de lote de materia
  prima coincide con una transición de proceso) que una relación física real de causa
  y efecto.

**Conclusión de la comprobación:** hay algo más que ruido puro a 60s, pero es débil,
descansa en muy pocas muestras, y la parte más "significativa" (Granger) coincide con
las variables que sabemos que son casi categóricas por cambios de lote — sospechoso de
ser coincidencia de régimen, no relación real. Aun así, probamos a entrenar el LSTM a
esta resolución para verlo a nivel de modelo, con expectativas moderadas dado lo poco
que hay: 165 filas de train, `SEQ_LEN=3` (3 min de historia) para no perder casi
ninguna fila en el windowing, y un modelo mucho más pequeño que en el resto del
notebook para no sobreajustar con tan pocos datos.

In [ ]:
df60 = df.resample('60s').mean()
for t, td in zip(MULTI_TARGETS, MULTI_TARGET_DIFFS):
    df60[td] = df60[t].diff()
df60 = df60.dropna(subset=MULTI_TARGET_DIFFS).copy()

feature_cols_60 = [c for c in df60.columns if c not in
                    (['time_stamp', TARGET, target_col] + MULTI_TARGETS + MULTI_TARGET_DIFFS)]

X_raw_60 = df60[feature_cols_60].values
Y_raw_60 = df60[MULTI_TARGET_DIFFS].values

n_60 = len(df60)
train_end_60 = int(n_60 * 0.70)
val_end_60 = int(n_60 * 0.85)

X_train_raw_60, Y_train_raw_60 = X_raw_60[:train_end_60], Y_raw_60[:train_end_60]
X_val_raw_60,   Y_val_raw_60   = X_raw_60[train_end_60:val_end_60], Y_raw_60[train_end_60:val_end_60]
X_test_raw_60,  Y_test_raw_60  = X_raw_60[val_end_60:], Y_raw_60[val_end_60:]

scaler_X_60 = MinMaxScaler()
X_train_60 = scaler_X_60.fit_transform(X_train_raw_60)
X_val_60   = scaler_X_60.transform(X_val_raw_60)
X_test_60  = scaler_X_60.transform(X_test_raw_60)

scaler_Y_60 = MinMaxScaler()
Y_train_60 = scaler_Y_60.fit_transform(Y_train_raw_60)
Y_val_60   = scaler_Y_60.transform(Y_val_raw_60)
Y_test_60  = scaler_Y_60.transform(Y_test_raw_60)

print(f"Filas totales (60s): {n_60}")
print(f"Train: {X_train_60.shape[0]} | Val: {X_val_60.shape[0]} | Test: {X_test_60.shape[0]}")
print(f"n_features: {X_train_60.shape[1]} | n_targets: {Y_train_60.shape[1]}")

In [ ]:
SEQ_LEN_60 = 3   # 3 pasos = 3 min de historia (a resolución de 60s/paso)
BATCH_SIZE_60 = 8  # muy pocos datos -> lotes pequeños

train_ds_60 = SequenceDataset(X_train_60, Y_train_60, SEQ_LEN_60)
val_ds_60   = SequenceDataset(X_val_60,   Y_val_60,   SEQ_LEN_60)
test_ds_60  = SequenceDataset(X_test_60,  Y_test_60,  SEQ_LEN_60)

train_loader_60 = DataLoader(train_ds_60, batch_size=BATCH_SIZE_60, shuffle=True)
val_loader_60   = DataLoader(val_ds_60,   batch_size=BATCH_SIZE_60, shuffle=False)
test_loader_60  = DataLoader(test_ds_60,  batch_size=BATCH_SIZE_60, shuffle=False)
train_eval_loader_60 = DataLoader(train_ds_60, batch_size=BATCH_SIZE_60, shuffle=False)

print(f"train windows: {len(train_ds_60)} | val windows: {len(val_ds_60)} | test windows: {len(test_ds_60)}")

# Modelo mucho más pequeño que en el resto del notebook: con ~160 ventanas de train,
# incluso el modelo de 32 unidades (10k parámetros) estaría muy sobreparametrizado.
n_features_60 = X_train_60.shape[1]
n_targets_60 = Y_train_60.shape[1]
model_60 = LSTMMultiRegressor(n_features=n_features_60, n_targets=n_targets_60,
                               hidden_size=8, num_layers=1, dropout=0.4).to(device)
print('Parámetros entrenables:', sum(p.numel() for p in model_60.parameters() if p.requires_grad))

In [ ]:
optimizer_60 = torch.optim.Adam(model_60.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler_60 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_60, mode='min', factor=0.5, patience=3)

train_losses_60, val_losses_60 = [], []
best_val_loss_60 = float('inf')
best_state_60 = None
epochs_no_improve_60 = 0

for epoch in range(1, N_EPOCHS + 1):
    train_loss = run_epoch(train_loader_60, model_60, criterion, optimizer_60)
    val_loss = run_epoch(val_loader_60, model_60, criterion, optimizer=None)
    scheduler_60.step(val_loss)

    train_losses_60.append(train_loss)
    val_losses_60.append(val_loss)

    improved = val_loss < best_val_loss_60
    if improved:
        best_val_loss_60 = val_loss
        best_state_60 = copy.deepcopy(model_60.state_dict())
        epochs_no_improve_60 = 0
    else:
        epochs_no_improve_60 += 1

    marker = " *" if improved else ""
    print(f"Epoch {epoch:3d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}{marker}")

    if epochs_no_improve_60 >= PATIENCE:
        print(f"Early stopping en epoch {epoch} (sin mejora en {PATIENCE} épocas).")
        break

model_60.load_state_dict(best_state_60)
print(f"\nMejor val_loss: {best_val_loss_60:.5f}")

In [ ]:
train_pred_s60, train_true_s60 = predict_multi(train_eval_loader_60, model_60)
val_pred_s60,   val_true_s60   = predict_multi(val_loader_60, model_60)
test_pred_s60,  test_true_s60  = predict_multi(test_loader_60, model_60)

zero_diff_scaled_60 = scaler_Y_60.transform(np.zeros((1, n_targets_60)))[0]

def naive_baseline_multi_60(Y_scaled, seq_len):
    y_hat = np.tile(zero_diff_scaled_60, (len(Y_scaled) - seq_len, 1))
    y_true = Y_scaled[seq_len:]
    return y_hat, y_true

train_naive_pred_s60, train_naive_true_s60 = naive_baseline_multi_60(Y_train_60, SEQ_LEN_60)
val_naive_pred_s60,   val_naive_true_s60   = naive_baseline_multi_60(Y_val_60,   SEQ_LEN_60)
test_naive_pred_s60,  test_naive_true_s60  = naive_baseline_multi_60(Y_test_60,  SEQ_LEN_60)

assert np.allclose(train_naive_true_s60, train_true_s60)
assert np.allclose(val_naive_true_s60, val_true_s60)
assert np.allclose(test_naive_true_s60, test_true_s60)

def inv_multi_60(Y_scaled):
    return scaler_Y_60.inverse_transform(Y_scaled)

results_60 = {
    'train': {'y_true': inv_multi_60(train_true_s60), 'lstm': inv_multi_60(train_pred_s60), 'naive': inv_multi_60(train_naive_pred_s60)},
    'val':   {'y_true': inv_multi_60(val_true_s60),   'lstm': inv_multi_60(val_pred_s60),   'naive': inv_multi_60(val_naive_pred_s60)},
    'test':  {'y_true': inv_multi_60(test_true_s60),  'lstm': inv_multi_60(test_pred_s60),  'naive': inv_multi_60(test_naive_pred_s60)},
}

rows_60 = []
for split in ['train', 'val', 'test']:
    for ti, target_name in enumerate(MULTI_TARGETS):
        y_true = results_60[split]['y_true'][:, ti]
        for model_name in ['lstm', 'naive']:
            y_pred = results_60[split][model_name][:, ti]
            m = compute_metrics(y_true, y_pred)
            m['split'] = split
            m['target'] = target_name.replace('FirstStage.CombinerOperation.', '')
            m['model'] = model_name
            rows_60.append(m)

metrics_df_60 = pd.DataFrame(rows_60).set_index(['split', 'target', 'model'])[['RMSE', 'MAE', 'MAPE (%)', 'R2']]
metrics_df_60 = metrics_df_60.round(4)
print(metrics_df_60.to_string())

print("\nSkill score (resolución 60s) por target y split:")
for split in ['train', 'val', 'test']:
    for target_name in MULTI_TARGETS:
        tname = target_name.replace('FirstStage.CombinerOperation.', '')
        rmse_lstm = metrics_df_60.loc[(split, tname, 'lstm'), 'RMSE']
        rmse_naive = metrics_df_60.loc[(split, tname, 'naive'), 'RMSE']
        skill = 1 - rmse_lstm / rmse_naive if rmse_naive != 0 else float('nan')
        print(f"  {split:5s} | {tname:15s}: skill={skill:+.4f}")

## Conclusión: LSTM a resolución de 60 segundos

Confirmado a nivel de modelo lo que ya anticipaba el test de permutación: con solo
~160 ventanas de entrenamiento, el LSTM no consigue superar la persistencia — de
hecho la pierde con claridad en los tres targets:

| Target | Skill score (test) |
|---|---|
| Temperature1 | -0.5% |
| Temperature2 | **-3247%** |
| Temperature3 | -0.7% |

`Temperature2` colapsa de forma especialmente dramática (R²=-1161 en test). La curva
de aprendizaje nunca llega a estabilizarse — sigue mejorando de forma irregular hasta
la época 50 sin converger, síntoma típico de entrenar con demasiado pocos datos para
el tamaño del problema, incluso con un modelo reducido a ~600 parámetros.

**Conclusión general de la línea "60 segundos":** remuestrear a 60s parecía prometedor
por la información mutua (MI) más alta, pero el test de permutación ya avisaba de que
gran parte de esa MI era un artefacto del tamaño de muestra pequeño (hasta la MI de
datos barajados al azar era ~10x la MI real encontrada a 1s). El entrenamiento del
LSTM confirma la sospecha: no hay suficientes datos en esta única campaña de ~4h para
que un split temporal 70/15/15 a 60s de resolución deje una muestra de train viable —
mismo patrón que ya vimos al probar 10s de resolución. La resolución de 1s con el
target diferenciado sigue siendo la configuración con mejor resultado (y la única
metodológicamente sólida) de todas las probadas en este notebook.